# TPC-H Nation & Region

**Dataset:** `samples.tpch.nation` + `samples.tpch.region`

**Difficulty:** Easy

**Topics:** join, groupBy, filter

In [0]:
from pyspark.sql import functions as F, types as T

## Learn — Joining DataFrames

| Operation | Syntax | What it does |
|-----------|--------|-------------|
| Inner join | `df1.join(df2, on_col, "inner")` | Keeps rows with matches in both tables |
| Left join | `df1.join(df2, on_col, "left")` | Keeps all rows from df1, nulls for non-matches in df2 |
| Join on expression | `df1.join(df2, df1.id == df2.fk, "inner")` | Join on columns with different names |
| Select after join | `.select("col1", "col2")` | Choose which columns to keep after join |
| `F.collect_list(col)` | Used with `.groupBy().agg(...)` | Collects all values in a group into an array |

**Docs:** [Join Types](https://spark.apache.org/docs/latest/sql-ref-syntax-qry-select-join.html) · [DataFrame.join()](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.join.html) · [PySpark Functions](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/functions.html)

In [0]:
# Run this example first — then solve the problems below.
# NOTE: this example is not a solution to any problem

nations = spark.table("samples.tpch.nation")
regions = spark.table("samples.tpch.region")

# Explore the schemas — see which columns link these tables
nations.printSchema()
regions.printSchema()

# Inner join: enrich each nation with its region name
enriched = nations.join(regions, nations.n_regionkey == regions.r_regionkey, "inner")
enriched.select("n_name", "r_name").show(5)

# collect_list: gather all nation names into one array per region
# (Problem 5 asks you to do this — here is the core pattern)
enriched.groupBy(F.col("r_name").alias("region")) \
        .agg(F.collect_list("n_name").alias("nations")) \
        .show(truncate=False)

## Problem 1

Join the nation table with the region table to produce a list of nations
along with their **region name**. Load `samples.tpch.nation` and
`samples.tpch.region`, join on `n_regionkey = r_regionkey`.
Sort by region name, then nation name.

**Expected output columns:**
- `n_name` - nation name
- `r_name` - region name (from the region table)

In [0]:
# Problem 1 - write your solution here
# Assign your result to: result_1
nations = spark.table("samples.tpch.nation")
regions = spark.table("samples.tpch.region")
result_1 = nations.join(
    regions,
    nations.n_regionkey == regions.r_regionkey,
    "inner"
).select("n_name", "r_name").orderBy(F.col("r_name"), F.col("n_name"))

In [0]:
# ── Tests for Problem 1 ──────────────────────────────────────────
assert result_1 is not None, "result_1 is None - did you forget to assign your DataFrame?"
assert hasattr(result_1, 'columns'), "result_1 must be a Spark DataFrame"
cols = [c.lower() for c in result_1.columns]
assert 'n_name' in cols, "Missing column: n_name"
assert 'r_name' in cols, "Missing column: r_name"
assert len(cols) == 2, f"Expected exactly 2 columns, got {len(cols)}: {cols}"
cnt = result_1.count()
assert cnt == 25, f"TPC-H has 25 nations, expected 25 rows, got {cnt}"
nation_names = [r['n_name'] for r in result_1.collect()]
assert len(set(nation_names)) == 25, "All 25 nation names must be distinct"
print(f"Problem 1 passed ✓  ({cnt} rows)")

## Problem 2

Count the number of **nations per region** using the joined data.
Join the two tables and group by region name.

**Expected output columns:**
- `r_name` - region name
- `nation_count` - number of nations in that region

In [0]:
# Problem 2 - write your solution here
# Assign your result to: result_2

result_2 = regions.join(
    nations,
    regions.r_regionkey == nations.n_regionkey,
    "inner"
).groupBy("r_name").agg(
    F.count("*").alias("nation_count")
)

In [0]:
display(result_2)

In [0]:
# ── Tests for Problem 2 ──────────────────────────────────────────
assert result_2 is not None, "result_2 is None - did you forget to assign your DataFrame?"
assert hasattr(result_2, 'columns'), "result_2 must be a Spark DataFrame"
cols = [c.lower() for c in result_2.columns]
assert 'r_name' in cols, "Missing column: r_name"
assert 'nation_count' in cols, "Missing column: nation_count"
assert len(cols) == 2, f"Expected exactly 2 columns, got {len(cols)}: {cols}"
cnt = result_2.count()
assert cnt == 5, f"TPC-H has 5 regions, expected 5 rows, got {cnt}"
total_nations = sum(r['nation_count'] for r in result_2.collect())
assert total_nations == 25, f"Total nation count across all regions must be 25, got {total_nations}"
print(f"Problem 2 passed ✓  ({cnt} rows)")

## Problem 3

Find all nations that belong to the **AFRICA** region.
Join the tables and filter where `r_name = 'AFRICA'`.

**Expected output columns:**
- `n_name` - nation name (nations in AFRICA only)

In [0]:
# Problem 3 - write your solution here
# Assign your result to: result_3

result_3 = nations.join(
    regions,
    nations.n_regionkey == regions.r_regionkey,
    "inner"
).filter(F.col("r_name") == "AFRICA").select("n_name")

In [0]:
# ── Tests for Problem 3 ──────────────────────────────────────────
assert result_3 is not None, "result_3 is None - did you forget to assign your DataFrame?"
assert hasattr(result_3, 'columns'), "result_3 must be a Spark DataFrame"
cols = [c.lower() for c in result_3.columns]
assert 'n_name' in cols, "Missing column: n_name"
assert len(cols) == 1, f"Expected exactly 1 columns, got {len(cols)}: {cols}"
cnt = result_3.count()
assert cnt == 5, f"TPC-H AFRICA region has 5 nations, expected 5 rows, got {cnt}"
nation_names = [r['n_name'] for r in result_3.collect()]
assert len(set(nation_names)) == cnt, "Nation names must be distinct"
print(f"Problem 3 passed ✓  ({cnt} rows)")

## Problem 4

Find all nations that belong to the **EUROPE** region.
Join the tables and filter where `r_name = 'EUROPE'`.

**Expected output columns:**
- `n_name` - nation name (nations in EUROPE only)

In [0]:
# Problem 4 - write your solution here
# Assign your result to: result_4

result_4 = nations.join(
    regions,
    nations.n_regionkey == regions.r_regionkey,
    "inner"
).filter(F.col("r_name") == "EUROPE").select("n_name")

In [0]:
# ── Tests for Problem 4 ──────────────────────────────────────────
assert result_4 is not None, "result_4 is None - did you forget to assign your DataFrame?"
assert hasattr(result_4, 'columns'), "result_4 must be a Spark DataFrame"
cols = [c.lower() for c in result_4.columns]
assert 'n_name' in cols, "Missing column: n_name"
assert len(cols) == 1, f"Expected exactly 1 columns, got {len(cols)}: {cols}"
cnt = result_4.count()
assert cnt == 5, f"TPC-H EUROPE region has 5 nations, expected 5 rows, got {cnt}"
nation_names = [r['n_name'] for r in result_4.collect()]
assert len(set(nation_names)) == cnt, "Nation names must be distinct"
print(f"Problem 4 passed ✓  ({cnt} rows)")

## Problem 5

For each region, collect all nation names into an **array column**.
Use `F.collect_list()` after joining and grouping by region name.

**Expected output columns:**
- `region` - region name
- `nations` - array of nation names belonging to that region

In [0]:
# Problem 5 - write your solution here
# Assign your result to: result_5

result_5 = regions.join(
    nations,
    regions.r_regionkey == nations.n_regionkey,
    "inner"
).groupBy(F.col("r_name").alias("region")).agg(
    F.collect_list("n_name").alias("nations")
)

In [0]:
display(result_5)

In [0]:
# ── Tests for Problem 5 ──────────────────────────────────────────
assert result_5 is not None, "result_5 is None - did you forget to assign your DataFrame?"
assert hasattr(result_5, 'columns'), "result_5 must be a Spark DataFrame"
cols = [c.lower() for c in result_5.columns]
assert 'region' in cols, "Missing column: region"
assert 'nations' in cols, "Missing column: nations"
assert len(cols) == 2, f"Expected exactly 2 columns, got {len(cols)}: {cols}"
cnt = result_5.count()
assert cnt == 5, f"Expected 5 rows (one per region), got {cnt}"
from pyspark.sql.types import ArrayType
nations_field = [f for f in result_5.schema.fields if f.name.lower() == 'nations'][0]
assert isinstance(nations_field.dataType, ArrayType), \
    f"'nations' column must be ArrayType, got {type(nations_field.dataType).__name__}"
rows = result_5.collect()
total_nations = sum(len(r['nations']) for r in rows)
assert total_nations == 25, f"Total nations across all regions must be 25, got {total_nations}"
assert all(len(r['nations']) == 5 for r in rows), "Each region must have exactly 5 nations"
print(f"Problem 5 passed ✓  ({cnt} rows)")